# Retina Age Prediction — full run on Colab

Adi Dror, Shahaf Gino | Deep Learning final project

**Before running:** make sure Runtime > Change runtime type > GPU is selected, and that the code (`src/`) is available here in Colab (see cell 2).

**How to use this notebook:** Runtime > Run all. Section 4 runs the whole experiment suite in one go; sections 5-9 are the same experiments one at a time, in case you'd rather step through them. Anything that already has a saved result is skipped automatically, so re-running after a disconnect never redoes finished work.

## 1. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Get the code

Three ways to get `src/` into this Colab session — pick ONE:

- **Zip upload (use this for now):** on your computer, zip the `Final Project` folder, then use the Files panel on the left (folder icon) to upload it here, and run the cell below to unzip it. Simplest, no login needed - the only downside is you re-upload it if the Colab runtime disconnects.
- **Google Drive:** `drive.mount(...)` currently has a known Colab bug where the permission screen loops or fails ("credential propagation was unsuccessful" - see https://github.com/googlecolab/colabtools/issues/5944). Skip this option until Google fixes it.
- **GitHub (best long-term option, once the project is pushed there):** comment out the unzip lines and uncomment the `git clone` line instead - no re-uploading ever again.

This cell also sets `HF_HUB_DISABLE_XET` **before** anything imports `huggingface_hub` - we hit a bug where the dataset download would hang at ~96% (a known issue with HuggingFace's "Xet" transfer backend), and this env var has to be set before the first import in the kernel to take effect.

In [ ]:
# Option A (default for now): unzip a Final Project.zip you uploaded via the Files panel
!unzip -q -o "Final Project.zip"
%cd "Final Project"

# Option B: once there's a GitHub repo, use this instead of the two lines above:
# !git clone https://github.com/<username>/<repo-name>.git repo
# %cd repo

import os
import sys

sys.path.append(".")  # assumes src/ is in the current working directory

# Must be set before huggingface_hub is imported anywhere below (dataset.py
# imports it lazily inside download_dataset(), so this is early enough) -
# works around the Xet download hang we hit earlier.
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
!pip install -q huggingface_hub

## 3. Download the dataset and sanity-check it before running everything

This dataset has no ready-made "image" column and no loading script, so it's downloaded directly: `images/` (9,857 JPEGs) + `splits/train.csv`, `val.csv`, `test.csv`. The first run downloads the ~10GB `images/` folder, which takes a while — it's cached locally, so re-running this cell later is instant (unless the Colab runtime disconnected, in which case it re-downloads).

**Important:** this also prints the CSV columns, to confirm they still match `src/dataset.py` (which looks for `image_id` and `patient_age`). If the dataset ever changes its column names, update them at the top of `src/dataset.py` before continuing.

**Why the first version of this was so slow.** Our first full run took ~10 minutes *per epoch*, and `nvidia-smi` showed the GPU at 0% utilization - it was idle, waiting on the CPU to decode ~1900x1900 JPEGs. The fixes now in `src/dataset.py`:

- **Resize once, not every epoch.** `get_dataloaders()` builds an `images_224/` cache the first time it runs, then trains from those small files. Measured: ~2.3s per batch of 32 before, ~0.3s after.
- **Faster cache build.** Uses PIL's `draft()` (lets libjpeg decode straight to a small size, ~2.6x faster) across a thread pool (~1.9x on Colab's 2 cores), so building the cache is a few minutes rather than ~15.
- **Fewer redundant checks.** A `.complete` marker means the cache isn't re-scanned by every experiment, and `download_dataset()` checks locally before asking HuggingFace anything (that redundant re-verification is what triggered the earlier `429 rate limited` warnings).

In [ ]:
from src.dataset import download_dataset, load_split_dataframe

root = download_dataset()
for split in ["train", "val", "test"]:
    df = load_split_dataframe(split, root)
    print(f"{split}: {len(df)} rows, columns: {list(df.columns)}")

In [ ]:
from src.dataset import get_dataloaders

# This first call builds the resized-image cache (see note above), so expect
# a one-time progress bar of a few minutes in a fresh session. Everything
# after it is fast.
train_loader, val_loader, test_loader = get_dataloaders(batch_size=4, augment_train=True)
imgs, ages = next(iter(train_loader))
print("Batch image shape:", imgs.shape)
print("Batch ages:", ages)

## 4. Run every experiment

One cell for the whole suite, in order: the linear baseline, the 3 CNN configs, and the 3 ablation experiments (2 models each) — 9 runs total.

- Anything already finished is **skipped** (its saved metrics are loaded instead), so this cell is safe to re-run at any point.
- Every experiment writes its metrics, plots and checkpoint the moment it finishes, and one experiment failing **does not stop the others** — whatever failed is listed at the end so it can be re-run on its own.
- Train/val loss and a per-epoch time are printed as it goes, so it's always obvious whether the model is learning and how long the rest will take.

Sections 5-8 below run the same experiments individually. After this cell they'll all just report "already have saved metrics", which is expected.

In [ ]:
from src.train import run_all_experiments

run_all_experiments()

## 5. Baseline: linear regression (non-deep)

Ridge regression on hand-crafted features (grayscale thumbnail + colour/brightness statistics) — the non-deep reference point every CNN below is compared against.

In [ ]:
from src.train import train_linear_baseline

linear_metrics = train_linear_baseline()
linear_metrics

## 6. CNN baseline (no regularization — expected to overfit)

Epoch count trimmed to 15 (from 30) once we measured the real per-epoch time — the overfitting pattern (train loss dropping while val loss flattens or rises) should already be clear well before epoch 15. This config deliberately has no early stopping: we *want* to watch it overfit.

In [ ]:
from src.train import run_cnn_baseline

cnn_baseline_metrics = run_cnn_baseline()
cnn_baseline_metrics

## 7. CNN + regularization (dropout + weight decay + early stopping)

In [ ]:
from src.train import run_cnn_regularized

cnn_reg_metrics = run_cnn_regularized()
cnn_reg_metrics

## 8. CNN + augmentation

In [ ]:
from src.train import run_cnn_augmented

cnn_aug_metrics = run_cnn_augmented()
cnn_aug_metrics

## 9. Ablation experiments (novelty / contribution)

The three experiments from the proposal — each compares two architecture configurations: receptive field (7x7 shallow vs 3x3 deep), padding (same vs valid), and dropout position (early conv layers vs late FC head). All three use early stopping, so each side stops on its own once it stops improving rather than always burning the full epoch budget.

In [ ]:
from src.train import run_ablation_receptive_field

shallow_7x7, deep_3x3 = run_ablation_receptive_field()

In [ ]:
from src.train import run_ablation_padding

same_pad, valid_pad = run_ablation_padding()

In [ ]:
from src.train import run_ablation_dropout_position

early_drop, late_drop = run_ablation_dropout_position()

## 10. Final comparison table + plots

Uses every metrics file saved so far to build the comparison table and the bar charts for the report.

In [ ]:
from src.evaluate import build_comparison_table

comparison_df = build_comparison_table()
comparison_df

## 11. Download the results

Run the next cell to download the `results/` folder (metrics + all plots). Worth doing as soon as the suite finishes — Colab wipes `/content` when the runtime disconnects, and this is the only copy of the numbers the report is built from.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("results", "zip", "results")
files.download("results.zip")